# Policy Gradient Template: Build REINFORCE Yourself

This notebook is a practice template.

You already studied the guided version:

```text
1 - Vanilla Policy Gradient (REINFORCE) [CartPole]
```

Now the goal is to rebuild the same algorithm yourself.

You will implement:

1. environment setup
2. policy network
3. action selection
4. discounted returns
5. policy update
6. one training episode
7. evaluation
8. full training loop
9. plotting
10. visualizing the trained policy

This notebook intentionally has TODOs. Fill them in one section at a time.


## 1) Imports

Use modern Gymnasium and regular PyTorch.

Important pieces:

- `gymnasium`: environment API
- `torch.nn`: neural network layers
- `Categorical`: sampling discrete actions from policy logits
- `matplotlib`: plotting rewards


In [ ]:
%pip install -U "gymnasium[classic-control]"

import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

print("gymnasium:", gym.__version__)
print("torch:", torch.__version__)


gymnasium: 1.3.0
torch: 2.10.0+cpu


## 2) Seeds and Device

TODO:

- set Python, NumPy, and PyTorch seeds
- create `device`

Hint:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```


In [2]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
    torch.cuda.manual_seed(SEED)
else:
    torch.manual_seed(SEED)
    device = "cpu"
    
print("device:", device)


device: cpu


## 3) Environments

Create two CartPole environments:

- `train_env`: used to collect experience and update the policy
- `test_env`: used to evaluate the policy

Also seed the action spaces.

Remember: modern Gymnasium uses `reset(seed=...)`, not `env.seed(...)`.


In [3]:
# TODO: create train_env using gym.make("CartPole-v1")
# TODO: create test_env using gym.make("CartPole-v1")

# TODO: seed train_env.action_space
# TODO: seed test_env.action_space

# TODO: reset train_env with seed and print one example state
# state, info = ...

# TODO: print observation_space and action_space

train_env = gym.make("CartPole-v1")
test_env = gym.make("CartPole-v1")

train_env.action_space.seed(SEED)
test_env.action_space.seed(SEED)

state, info = train_env.reset(seed=SEED)

print(f"Example state is {state}, observation space is {train_env.observation_space} and the action space is {train_env.action_space}")



Example state is [ 0.04766998 -0.01198043  0.04232462 -0.02383076], observation space is Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32) and the action space is Discrete(2)


## 4) Policy Network

Build a small MLP policy.

CartPole observation:

```text
4 numbers
```

CartPole actions:

```text
2 actions: left or right
```

The network should map:

```text
state -> action logits
```

Do not apply softmax in the model. Return raw logits.


In [6]:
class PolicyNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Linear(
                in_features=input_dim,
                out_features=hidden_dim
                ),
            nn.ReLU(),
            nn.Linear(
                in_features=hidden_dim,
                out_features=output_dim
                ),  
        )
    
    def forward(self, x):
        return self.block1(x)




## 5) Instantiate Policy and Optimizer

Use the environment to determine dimensions:

```python
INPUT_DIM = train_env.observation_space.shape[0]
OUTPUT_DIM = train_env.action_space.n
```

Then create:

- `policy`
- `optimizer`


In [8]:
# TODO: get output dimension from train_env

INPUT_DIM = train_env.observation_space.shape[0]
OUTPUT_DIM = train_env.action_space.n

print(INPUT_DIM, OUTPUT_DIM)

HIDDEN_DIM = 128

policy0 = PolicyNetwork(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)

LEARNING_RATE = 1e-2

# TODO: create Adam optimizer
optimizer = optim.Adam(
    params=policy0.parameters(),
    lr = LEARNING_RATE
)

print(policy0)
print(policy0.state_dict())


4 2
PolicyNetwork(
  (block1): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)
OrderedDict({'block1.0.weight': tensor([[-0.2338, -0.3829, -0.4234, -0.1804],
        [ 0.1247, -0.0105,  0.0172,  0.2099],
        [ 0.1429,  0.3361, -0.1198,  0.1147],
        [-0.0050, -0.1716, -0.3356, -0.1386],
        [-0.4732,  0.0593, -0.3974, -0.1739],
        [-0.1954,  0.4285, -0.4076, -0.1194],
        [ 0.1790,  0.1414,  0.4103,  0.1213],
        [ 0.2939, -0.3001,  0.3104,  0.0109],
        [-0.0555, -0.2244,  0.1853,  0.2588],
        [-0.3660, -0.3866,  0.1995, -0.4425],
        [-0.4703,  0.3617,  0.1591,  0.0385],
        [-0.0029,  0.2077, -0.3571,  0.4157],
        [-0.0697,  0.0491, -0.4733,  0.0304],
        [-0.3055,  0.2813, -0.4750,  0.4507],
        [-0.1172, -0.1706, -0.0032, -0.4087],
        [ 0.1613,  0.4078, -0.2100, -0.1624],
        [-0.4166,  0.1218,  0.3683,  0.4070],
 

## 6) Optional: Initialize Weights

This is not the main RL idea, but it can help training.

TODO:

- initialize linear layer weights with Xavier normal
- initialize biases with zeros
- apply the initializer to the policy


In [ ]:
def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight)
        nn.init.zeros_(module.bias)


policy0.apply(init_weights)


## 7) Select an Action

This function should:

1. convert `state` to a tensor
2. pass it through the policy to get logits
3. create `Categorical(logits=logits)`
4. sample an action
5. calculate `log_prob` of that action
6. return `action.item()` and `log_prob`

Why store `log_prob`?

Because REINFORCE uses it later in the loss:

```python
loss = -(returns * log_probs).sum()
```


In [ ]:
def select_action(policy, state):
    # TODO: convert state to float tensor on device and add batch dimension
    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device)

    # TODO: get logits from policy
    logits = policy0(state_tensor)

    # TODO: create categorical distribution from logits
    distribution = Categorical(logits=logits)

    # TODO: sample action
    action = distribution.sample()

    # TODO: get log probability of selected action
    log_prob = distribution.log_prob(action)

    # TODO: return Python int action and squeezed log_prob tensor
    return action.item(), log_prob

tensor([-0.5498], grad_fn=<UnsqueezeBackward0>)
1
tensor(-0.5498, grad_fn=<SqueezeBackward1>)


## 8) Calculate Discounted Returns

Given rewards from one episode:

```text
[r0, r1, r2, ...]
```

calculate:

```text
G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
```

TODO:

- loop through rewards backwards
- build a list of returns
- convert to tensor
- optionally normalize returns


In [ ]:
def calculate_returns(rewards, discount_factor, normalize=True):
    returns = []
    running_return = 0.0

    # TODO: loop through rewards in reverse
    # TODO: update running_return
    # TODO: insert running_return at the beginning of returns

    # TODO: convert returns to tensor on device
    # returns = ...

    if normalize and len(returns) > 1:
        # TODO: normalize returns safely
        pass

    return returns


## 9) Update Policy

This is where learning happens.

REINFORCE loss:

```python
loss = -(returns * log_probs).sum()
```

TODO:

- stack `log_probs`
- detach `returns`
- compute loss
- zero gradients
- backprop
- optimizer step
- return `loss.item()`


In [ ]:
def update_policy(returns, log_probs, optimizer):
    # TODO: detach returns
    # TODO: stack log_probs
    # TODO: compute policy gradient loss
    # TODO: zero gradients
    # TODO: backward
    # TODO: optimizer step
    # TODO: return loss.item()
    pass


## 10) Train for One Episode

This is the most important function in the template.

It should:

1. reset the environment
2. run until `terminated or truncated`
3. use `select_action` to choose actions
4. call `env.step(action)`
5. save rewards and log probabilities
6. calculate returns
7. update the policy once after the episode ends

This is vanilla REINFORCE: one update per full episode.


In [ ]:
def train_one_episode(env, policy, optimizer, discount_factor, seed=None):
    policy.train()

    log_probs = []
    rewards = []
    episode_reward = 0.0

    # TODO: reset environment with seed
    # state, info = ...

    terminated = False
    truncated = False

    while not (terminated or truncated):
        # TODO: select action and log_prob
        # TODO: step environment
        # TODO: append log_prob
        # TODO: append reward
        # TODO: add reward to episode_reward
        # TODO: move state forward
        pass

    # TODO: calculate returns
    # TODO: update policy

    # TODO: return loss and episode_reward
    pass


## 11) Evaluate Policy

Evaluation should not train the policy.

Use `policy.eval()` and `torch.no_grad()`.

For evaluation, choose the most likely action:

```python
action = torch.argmax(logits, dim=-1).item()
```


In [ ]:
def evaluate(env, policy, seed=None):
    policy.eval()

    episode_reward = 0.0

    # TODO: reset environment
    # state, info = ...

    terminated = False
    truncated = False

    while not (terminated or truncated):
        # TODO: convert state to tensor
        # TODO: get logits with torch.no_grad()
        # TODO: choose argmax action
        # TODO: step environment
        # TODO: add reward
        pass

    return episode_reward


## 12) Full Training Loop

Now combine everything.

TODO:

- train for up to `MAX_EPISODES`
- evaluate after each episode
- store train/test rewards and losses
- print progress every `PRINT_EVERY` episodes
- stop when recent average test reward reaches threshold


In [ ]:
MAX_EPISODES = 500
DISCOUNT_FACTOR = 0.99
N_TRIALS = 25
REWARD_THRESHOLD = 475
PRINT_EVERY = 10

train_rewards = []
test_rewards = []
losses = []
recent_test_rewards = deque(maxlen=N_TRIALS)

for episode in range(1, MAX_EPISODES + 1):
    # TODO: call train_one_episode
    # loss, train_reward = ...

    # TODO: call evaluate
    # test_reward = ...

    # TODO: append loss, train_reward, test_reward
    # TODO: update recent_test_rewards

    # TODO: calculate recent means

    # TODO: print progress every PRINT_EVERY episodes

    # TODO: stop if solved
    pass


## 13) Plot Rewards

Plot train and test rewards.

Question:

- Does the policy improve over time?
- Is the curve smooth or noisy?


In [ ]:
# TODO: plot test_rewards
# TODO: plot train_rewards
# TODO: add labels, legend, grid, and threshold line


## 14) Plot Loss

Policy-gradient loss can be noisy.

Question:

- Does the loss tell the story as clearly as reward does?


In [ ]:
# TODO: plot losses


## 15) Watch the Trained Policy

Create a separate environment with `render_mode="human"` to open a window.

Do not render during training because it slows everything down.

If the window closes quickly, the policy may not be trained yet or the episode ended quickly.


In [ ]:
def watch_policy(policy, seed=SEED, max_steps=500):
    render_env = gym.make("CartPole-v1", render_mode="human")

    # TODO: reset render_env
    # TODO: run one episode using argmax actions from the policy
    # TODO: print total reward and steps
    # TODO: close render_env
    pass


# TODO: uncomment after training
# watch_policy(policy)


## 16) Self-Check

You are done when you can explain each line of this flow:

```text
state -> policy -> logits -> Categorical -> action
state/action -> env.step -> reward/next_state
rewards -> returns
returns/log_probs -> loss
loss.backward -> better policy
```

Questions to answer:

1. Why do we sample during training?
2. Why do we use argmax during evaluation?
3. Why do we store `log_prob`?
4. Why do we calculate returns after the episode ends?
5. Why is the REINFORCE loss negative?
